# Standalone ExDark LLIE Enhancement (HVI-CIDNet Generalization)

Notebook ini dirancang mandiri untuk Kaggle dengan dukungan **Multi-GPU (2x T4)** untuk pemrosesan super cepat.

### Cara Menyiapkan Weights:
1. Klik **File** -> **Upload Input** -> **Upload Dataset**.
2. Masukkan file `pytorch_model.bin` atau `model.safetensors` anda.
3. Beri nama bebas (misal: 'hvi-weights').
4. Klik **Create**.

**Note**: Script ini akan **mencari file tersebut secara otomatis** di semua folder input.


## 1. Setup & Install Dependencies


In [ ]:
!pip install -q safetensors tqdm opencv-python
!git clone https://github.com/Fediory/HVI-CIDNet.git


## 2. Configuration & Auto-Discovery

Script ini secara otomatis mencari dataset ExDark dan weights Anda.


In [ ]:
import os
import sys
import json
import glob
import numpy as np

# === 2.1 Auto-Create config.json ===
hvi_config = {"channels": [36, 36, 72, 144], "heads": [1, 2, 4, 8], "norm": False}
with open("config.json", "w") as f:
    json.dump(hvi_config, f)

# === 2.2 Discovery Logic ===
def find_file(filename, root="/kaggle/input"):
    paths = glob.glob(os.path.join(root, "**", filename), recursive=True)
    if not paths:
        paths = glob.glob(os.path.join("/kaggle/working", "**", filename), recursive=True)
    return paths[0] if paths else None

classlist_path = find_file("imageclasslist.txt")
if classlist_path:
    GT_DIR = os.path.dirname(classlist_path)
    ds_root = os.path.dirname(GT_DIR) 
    IMG_DIR = os.path.join(ds_root, "Dataset", "Dataset") if os.path.exists(os.path.join(ds_root, "Dataset", "Dataset")) else os.path.join(ds_root, "Dataset")
    print(f"Dataset found at: {ds_root}")
else:
    print("CRITICAL: Dataset ExDark tidak ditemukan.")
    IMG_DIR = ""
    GT_DIR = ""

WEIGHTS_PATH = find_file("pytorch_model.bin") or find_file("model.safetensors")
if WEIGHTS_PATH:
    print(f"Weights found at: {WEIGHTS_PATH} ✓")
else:
    print("CRITICAL: File weights tidak ditemukan!")

CONFIG_PATH = "config.json"
OUTPUT_ENHANCED_YOLO = "/kaggle/working/ExDark_Enhanced"


## 3. Dataset Splitting


In [ ]:
CLASS_ID_TO_FOLDER = {
    1: "Bicycle", 2: "Boat", 3: "Bottle", 4: "Bus",
    5: "Car", 6: "Cat", 7: "Chair", 8: "Cup",
    9: "Dog", 10: "Motorbike", 11: "People", 12: "Table",
}
SPLIT_MAP = {1: "train", 2: "val", 3: "test"}
images_split = {"train": [], "val": [], "test": []}

if GT_DIR and os.path.exists(os.path.join(GT_DIR, "imageclasslist.txt")):
    with open(os.path.join(GT_DIR, "imageclasslist.txt"), 'r') as f:
        for line in f:
            if line.startswith(("Name", "#", "%")): continue
            parts = line.strip().split()
            if len(parts) >= 5:
                fname, cid, _, _, split_id = parts[:5]
                if int(cid) in CLASS_ID_TO_FOLDER and int(split_id) in SPLIT_MAP:
                    images_split[SPLIT_MAP[int(split_id)]].append((fname, CLASS_ID_TO_FOLDER[int(cid)]))

print("Rasio Terdeteksi:")
for k, v in images_split.items():
    print(f"  {k}: {len(v)}")


## 4. Multi-GPU Model Loader & Worker


In [ ]:
import cv2
import torch
import torch.multiprocessing as mp
from tqdm.auto import tqdm
from safetensors.torch import load_file

if "HVI-CIDNet" not in sys.path:
    sys.path.insert(0, "HVI-CIDNet")
from net.CIDNet import CIDNet

# Load model di GPU tertentu
def load_model_on_device(device_id):
    device = torch.device(f'cuda:{device_id}' if torch.cuda.is_available() else 'cpu')
    with open(CONFIG_PATH, 'r') as f:
        config = json.load(f)
    model = CIDNet(**config)
    if WEIGHTS_PATH:
        if WEIGHTS_PATH.endswith('.safetensors'):
            state_dict = load_file(WEIGHTS_PATH)
        else:
            checkpoint = torch.load(WEIGHTS_PATH, map_location='cpu')
            state_dict = checkpoint["state_dict"] if "state_dict" in checkpoint else (checkpoint["model"] if "model" in checkpoint else checkpoint)
        model.load_state_dict(state_dict)
    model.to(device).eval()
    return model, device

# Proses Enhancement vram-safe
def enhance_image_vram_safe(model, device, img_bgr):
    h, w = img_bgr.shape[:2]
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    img_tensor = torch.from_numpy(img_rgb).float().permute(2, 0, 1).unsqueeze(0) / 255.0
    img_tensor = img_tensor.to(device)
    ph, pw = (8 - h % 8) % 8, (8 - w % 8) % 8
    img_tensor = torch.nn.functional.pad(img_tensor, (0, pw, 0, ph), mode='reflect')
    with torch.no_grad():
        out = model(img_tensor)
        if isinstance(out, (list, tuple)): out = out[0]
    out = out[:, :, :h, :w].squeeze(0).clamp(0, 1).cpu()
    del img_tensor
    torch.cuda.empty_cache()
    out_np = out.numpy()
    res = cv2.cvtColor((np.transpose(out_np, (1, 2, 0)) * 255).astype(np.uint8), cv2.COLOR_RGB2BGR)
    return res

# Worker Process
def worker_process(device_id, image_subset, split_name):
    model, device = load_model_on_device(device_id)
    out_dir = os.path.join(OUTPUT_ENHANCED_YOLO, "images", split_name)
    desc = f"GPU {device_id} [{split_name}]"
    for fname, folder in tqdm(image_subset, desc=desc, position=device_id, leave=False):
        src = os.path.join(IMG_DIR, folder, fname)
        dst = os.path.join(out_dir, fname)
        if os.path.exists(dst) or not os.path.exists(src): continue
        try:
            img = cv2.imread(src)
            if img is not None:
                enhanced = enhance_image_vram_safe(model, device, img)
                cv2.imwrite(dst, enhanced)
        except Exception as e:
            print(f"\n[GPU {device_id}] Error {fname}: {str(e)[:100]}")


## 5. Execute Enhancement (Multi-GPU)


In [ ]:
os.makedirs(OUTPUT_ENHANCED_YOLO, exist_ok=True)
num_gpus = torch.cuda.device_count()
print(f"Detected {num_gpus} GPUs. Starting Parallel Processing...")

for split in ["train", "val", "test"]:
    files = images_split[split]
    if not files: continue
    os.makedirs(os.path.join(OUTPUT_ENHANCED_YOLO, "images", split), exist_ok=True)
    
    # Split data per GPU
    chunks = np.array_split(files, num_gpus)
    processes = []
    for i in range(num_gpus):
        p = mp.Process(target=worker_process, args=(i, chunks[i].tolist(), split))
        p.start()
        processes.append(p)
    for p in processes:
        p.join()

print("\nEnhancement Complete for all splits!")


## 6. Zipping Output

Eksekusi sel ini untuk mengemas hasil menjadi file `.zip`.


In [ ]:
import shutil
print("Zipping... (Harap tunggu, proses ini memakan waktu beberapa menit)")
shutil.make_archive("/kaggle/working/ExDark_Enhanced_HVICIDNet_Gen", 'zip', OUTPUT_ENHANCED_YOLO)
print("\nDONE! Zip created at: /kaggle/working/ExDark_Enhanced_HVICIDNet_Gen.zip")
